In [64]:
import pandas as pd

## EDA

In [65]:
df = pd.read_csv('data/marketplace_products_combined_11k.csv')
df.head()


,platform,category,product_id,product_name,price,original_price,discount_percent,currency,rating,review_count,sold_count,seller_location,sold_count_estimate,sold_count_is_lower_bound,price_outlier,record_origin
0,tiki,Làm đẹp,630777,Kem chống nắng Sunplay cực mạnh dạng sữa Sunpl...,167000.0,NaN,NaN,VND,4.8,425.0,NaN,NaN,NaN,NaN,False,crawled
1,tiki,Làm đẹp,1313873,Kem chống nắng Sunplay mát lạnh giải nhiệt dạn...,86000.0,NaN,NaN,VND,4.8,221.0,NaN,NaN,NaN,NaN,False,crawled
2,tiki,Làm đẹp,45915523,Kem Chống Nắng Clarins UV Plus Multi Protectio...,1090000.0,NaN,NaN,VND,NaN,NaN,NaN,NaN,NaN,NaN,True,crawled
3,tiki,Làm đẹp,251536647,Kem chống nắng TENAMYD SPF50/PA+++,NaN,NaN,NaN,VND,NaN,NaN,NaN,NaN,NaN,NaN,NaN,crawled
4,tiki,Làm đẹp,248355431,Kem Chống Nắng Kiềm Dầu Farmona Sun Face Cream...,NaN,NaN,NaN,VND,NaN,NaN,NaN,NaN,NaN,NaN,NaN,crawled


In [66]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11050 entries, 0 to 11049
Data columns (total 16 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   platform                   11050 non-null  object 
 1   category                   11050 non-null  object 
 2   product_id                 11050 non-null  object 
 3   product_name               11050 non-null  object 
 4   price                      10747 non-null  float64
 5   original_price             3367 non-null   float64
 6   discount_percent           8148 non-null   float64
 7   currency                   11050 non-null  object 
 8   rating                     3110 non-null   float64
 9   review_count               3310 non-null   float64
 10  sold_count                 8742 non-null   object 
 11  seller_location            5955 non-null   object 
 12  sold_count_estimate        8742 non-null   float64
 13  sold_count_is_lower_bound  8742 non-null   obj

In [67]:
# df.drop(columns=['brand', 'search_keyword', 'seller_name', 'seller_rating', 'availability', 'product_url', 'crawl_time_utc'], inplace = True)

In [68]:
df['rating'].value_counts()

rating
5.0    1152
4.9     691
4.8     550
4.7     336
4.6     150
4.5      80
4.4      31
3.0      22
4.1      16
4.0      13
4.3      12
4.2      11
3.9       9
3.5       6
3.2       6
3.7       5
3.3       4
3.6       4
3.8       4
3.1       4
3.4       3
1.0       1
Name: count, dtype: int64

In [69]:
# Tóm tắt cấu trúc và số lượng theo sàn
print(f"Số dòng: {len(df):,} | Số cột: {df.shape[1]}")
display(df.groupby("platform").size().rename("product_count").to_frame())
display(pd.crosstab(df["platform"], df["category"]))

Số dòng: 11,050 | Số cột: 16


,product_count
platform,
lazada,3574
shopee,3771
tiki,3705


category,Làm đẹp,Nhà cửa & đời sống,Sức khỏe,Thời trang nữ,Thời trang trẻ em,Thực phẩm & đồ uống,Văn phòng phẩm,Điện gia dụng,Điện tử
platform,,,,,,,,,
lazada,401,407,381,394,381,397,403,406,404
shopee,368,388,715,403,368,361,387,368,413
tiki,518,506,285,545,276,422,342,324,487


In [70]:
# Tỷ lệ null toàn bộ và theo sàn
missing = pd.DataFrame({
    "null_count": df.isna().sum(),
    "null_pct": (df.isna().mean() * 100).round(1),
}).sort_values("null_pct", ascending=False)
display(missing)
missing_by_platform = pd.concat(
    {platform: group.isna().mean().mul(100).round(1) for platform, group in df.groupby("platform")},
    axis=1,
).T
display(missing_by_platform)

,null_count,null_pct
rating,7940,71.9
review_count,7740,70.0
original_price,7683,69.5
seller_location,5095,46.1
discount_percent,2902,26.3
sold_count,2308,20.9
sold_count_estimate,2308,20.9
sold_count_is_lower_bound,2308,20.9
price,303,2.7
price_outlier,303,2.7


,platform,category,product_id,product_name,price,original_price,discount_percent,currency,rating,review_count,sold_count,seller_location,sold_count_estimate,sold_count_is_lower_bound,price_outlier,record_origin
lazada,0.0,0.0,0.0,0.0,0.0,100.0,15.4,0.0,100.0,9.9,9.9,26.6,9.9,9.9,0.0,0.0
shopee,0.0,0.0,0.0,0.0,1.4,10.7,1.0,0.0,19.9,100.0,16.1,11.7,16.1,16.1,1.4,0.0
tiki,0.0,0.0,0.0,0.0,6.8,100.0,62.5,0.0,97.6,97.6,36.3,100.0,36.3,36.3,6.8,0.0


In [71]:
# Kiểm tra duplicate theo dòng, theo khóa sản phẩm và theo URL
print("Duplicate toàn dòng:", df.duplicated().sum())
print("Duplicate (platform, product_id):", df.duplicated(["platform", "product_id"]).sum())
if "product_url" in df.columns:
    print("Duplicate (platform, product_url):", df.duplicated(["platform", "product_url"]).sum())

Duplicate toàn dòng: 0
Duplicate (platform, product_id): 0


In [72]:
# Phân phối các chỉ số dạng số; bản gốc không bị ghi đè
numeric_cols = [c for c in ["price", "original_price", "discount_percent", "rating", "review_count"] if c in df.columns]
numeric = df[numeric_cols].apply(pd.to_numeric, errors="coerce")
display(numeric.describe(percentiles=[.01, .25, .5, .75, .95, .99]).T)
if "price" in numeric:
    print("Giá bằng 0:", numeric["price"].eq(0).sum())
if "rating" in numeric:
    invalid_rating = numeric["rating"].notna() & ~numeric["rating"].between(0, 5)
    print("Rating ngoài [0, 5]:", invalid_rating.sum())
    display(df.loc[invalid_rating, ["platform", "product_id", "product_name", "rating"]].head(20))

,count,mean,std,min,1%,25%,50%,75%,95%,99%,max
price,10747.0,430821.195869,8.626141e+05,51.0,8000.0,71000.0,165000.0,410000.0,1670000.0,4790800.00,12965000.0
original_price,3367.0,586545.196614,1.049313e+06,500.0,27830.0,140000.0,270000.0,540000.0,2400000.0,5453800.00,16200000.0
discount_percent,8148.0,35.253191,1.950266e+01,1.0,1.0,20.0,35.0,48.0,71.0,85.00,100.0
rating,3110.0,4.820932,2.875274e-01,1.0,3.2,4.8,4.9,5.0,5.0,5.00,5.0
review_count,3310.0,687.515106,2.268963e+03,0.0,0.0,9.0,49.0,347.0,3481.5,9631.71,48201.0


Giá bằng 0: 0
Rating ngoài [0, 5]: 0


,platform,product_id,product_name,rating


In [73]:
# Gắn cờ ngoại lệ giá theo từng sàn và danh mục; không xóa hay winsorize
if "price" in df.columns:
    price_check = df[["platform", "category", "price"]].copy()
    price_check["category"] = price_check["category"].replace({"health": "Sức khỏe"})
    price_check["price"] = pd.to_numeric(price_check["price"], errors="coerce").replace(0, pd.NA)
    summaries = []
    for (platform, category), group in price_check.groupby(["platform", "category"], dropna=False):
        x = group["price"].dropna()
        if x.empty:
            continue
        q1, q3 = x.quantile([.25, .75])
        iqr = q3 - q1
        low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        summaries.append({"platform": platform, "category": category, "n": x.size, "q1": q1, "median": x.median(), "q3": q3, "lower_fence": low, "upper_fence": high, "outside_iqr_fences": ((x < low) | (x > high)).sum()})
    display(pd.DataFrame(summaries).set_index(["platform", "category"]))

n          q1     median         q3  \
platform category                                                     
lazada   Làm đẹp              401    75500.00   150000.0   323000.0   
         Nhà cửa & đời sống   407    63000.00   135000.0   210000.0   
         Sức khỏe             381   124500.00   290000.0   483000.0   
         Thời trang nữ        394    40000.00    54000.0    74000.0   
         Thời trang trẻ em    381    36500.00    63000.0    90000.0   
         Thực phẩm & đồ uống  397   110000.00   165000.0   240000.0   
         Văn phòng phẩm       403    16600.00    55000.0   101000.0   
         Điện gia dụng        406   600000.00   800000.0  1030000.0   
         Điện tử              404    60000.00    98000.0   300000.0   
shopee   Làm đẹp              368   164625.00   239500.0   340000.0   
         Nhà cửa & đời sống   388   101750.00   196750.0   314643.0   
         Sức khỏe             715    75000.00   151000.0   250500.0   
         Thời trang nữ        403    63500.00    96500.0   149000.0   
         Thời trang trẻ em    368    55000.00    80000.0   105500.0   
         Thực phẩm & đồ uống  361    57000.00   112000.0   210000.0   
         Văn phòng phẩm       336    14082.25    30750.0   106000.0   
         Điện gia dụng        368   809750.00  1352500.0  1952500.0   
         Điện tử              413   169000.00   261000.0   560000.0   
tiki     Làm đẹp              494   193875.00   335000.0   560000.0   
         Nhà cửa & đời sống   447    90000.00   255000.0   584000.0   
         Sức khỏe             285   134000.00   560000.0   960000.0   
         Thời trang nữ        545    60000.00   114000.0   185000.0   
         Thời trang trẻ em    276   103125.00   168000.0   260000.0   
         Thực phẩm & đồ uống  422    80000.00   150000.0   254000.0   
         Văn phòng phẩm       250    37250.00    59500.0   577500.0   
         Điện gia dụng        254  1495000.00  2230000.0  2873750.0   
         Điện tử              480   390000.00   672500.0  2268750.0   

                              lower_fence  upper_fence  outside_iqr_fences  
platform category                                                           
lazada   Làm đẹp              -295750.000   694250.000                  33  
         Nhà cửa & đời sống   -157500.000   430500.000                  23  
         Sức khỏe             -413250.000  1020750.000                   9  
         Thời trang nữ         -11000.000   125000.000                  28  
         Thời trang trẻ em     -43750.000   170250.000                   3  
         Thực phẩm & đồ uống   -85000.000   435000.000                  34  
         Văn phòng phẩm       -110000.000   227600.000                  19  
         Điện gia dụng         -45000.000  1675000.000                  20  
         Điện tử              -300000.000   660000.000                  39  
shopee   Làm đẹp               -98437.500   603062.500                  14  
         Nhà cửa & đời sống   -217589.500   633982.500                  36  
         Sức khỏe             -188250.000   513750.000                  28  
         Thời trang nữ         -64750.000   277250.000                  20  
         Thời trang trẻ em     -20750.000   181250.000                  15  
         Thực phẩm & đồ uống  -172500.000   439500.000                  22  
         Văn phòng phẩm       -123794.375   243876.625                  40  
         Điện gia dụng        -904375.000  3666625.000                  19  
         Điện tử              -417500.000  1146500.000                  30  
tiki     Làm đẹp              -355312.500  1109187.500                  38  
         Nhà cửa & đời sống   -651000.000  1325000.000                  31  
         Sức khỏe            -1105000.000  2199000.000                   0  
         Thời trang nữ        -127500.000   372500.000                  22  
         Thời trang trẻ em    -132187.500   495312.500                   1  
         Thực phẩm & đồ uống  -181000.000   515

## Tiền xử lý 
Mỗi cell dưới đây làm một bước và tạo `df_clean` từ `df`. Các giá trị null được giữ nguyên, không điền median. CSV nguồn không bị ghi đè. Chạy lần lượt các cell theo thứ tự.

In [74]:
# 1. Tạo bản sao làm sạch và chuẩn hóa nhãn danh mục
df_clean = df.copy()
df_clean["category"] = df_clean["category"].astype("string").str.strip()
df_clean["category"] = df_clean["category"].replace({"health": "Sức khỏe"})
print("Số danh mục health đã chuẩn hóa:", int(df["category"].astype("string").str.strip().eq("health").sum()))

Số danh mục health đã chuẩn hóa: 0


In [75]:
# 2. Chuẩn hóa kiểu số và xử lý các giá trị không hợp lệ
import numpy as np
for col in ["price", "original_price", "discount_percent", "rating", "review_count"]:
    df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")

# Giá 0 không hợp lệ trong dữ liệu sản phẩm này; chuyển thành null.
zero_price = df_clean["price"].eq(0)
df_clean.loc[zero_price, "price"] = np.nan

# Rating ngoài [0, 5] được xem là lỗi parse và chuyển thành null.
invalid_rating = df_clean["rating"].notna() & ~df_clean["rating"].between(0, 5)
df_clean.loc[invalid_rating, "rating"] = np.nan

# Toàn bộ dữ liệu thuộc các sàn Việt Nam; điền VND nếu currency trống.
currency = df_clean["currency"].astype("string").str.strip()
currency = currency.mask(currency.eq(""), pd.NA)
df_clean["currency"] = currency.fillna("VND")
print("Giá 0 chuyển null:", int(zero_price.sum()))
print("Rating ngoài [0, 5] chuyển null:", int(invalid_rating.sum()))

Giá 0 chuyển null: 0
Rating ngoài [0, 5] chuyển null: 0


In [76]:
# 3. Giữ lượt bán gốc và tạo ước lượng số lượt bán
import re
def parse_sold_count(value):
    if pd.isna(value):
        return np.nan, pd.NA

    text = str(value).strip()
    match = re.fullmatch(r"(\d+(?:[.,]\d+)?)\s*([kKmM])?\s*(\+)?", text)
    if not match:
        return np.nan, pd.NA

    amount = float(match.group(1).replace(",", "."))
    suffix = (match.group(2) or "").lower()
    multiplier = {"": 1, "k": 1_000, "m": 1_000_000}[suffix]
    estimate = round(amount * multiplier)
    is_lower_bound = bool(match.group(3))
    return estimate, is_lower_bound

parsed_sold = df_clean["sold_count"].apply(parse_sold_count)
df_clean["sold_count_estimate"] = parsed_sold.map(lambda result: result[0])
df_clean["sold_count_is_lower_bound"] = pd.array(
    parsed_sold.map(lambda result: result[1]), dtype="boolean"
)
print("Lượt bán quy đổi được:", int(df_clean["sold_count_estimate"].notna().sum()))
print("Ước lượng là cận dưới:", int(df_clean["sold_count_is_lower_bound"].sum(skipna=True)))

Lượt bán quy đổi được: 8742
Ước lượng là cận dưới: 1125


In [77]:
# 4. Gắn cờ ngoại lệ giá theo từng sàn và danh mục
df_clean["price_outlier"] = pd.Series(
    pd.NA, index=df_clean.index, dtype="boolean"
)

for _, group in df_clean.groupby(["platform", "category"], dropna=False):
    prices = group["price"].dropna()
    if prices.empty:
        continue

    q1, q3 = prices.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower_fence = q1 - 1.5 * iqr
    upper_fence = q3 + 1.5 * iqr
    flags = prices.lt(lower_fence) | prices.gt(upper_fence)
    df_clean.loc[prices.index, "price_outlier"] = flags.to_numpy()

print("Giá được gắn cờ ngoại lệ:", int(df_clean["price_outlier"].sum()))

Giá được gắn cờ ngoại lệ: 635


In [78]:
# 5. Xuất file mới và xem nhanh kết quả
output_path = "data/marketplace_products_clean.csv"
df_clean.to_csv(output_path, index=False, encoding="utf-8-sig")

display(df_clean.groupby("platform").size().rename("product_count").to_frame())
display(df_clean[["price", "original_price", "discount_percent", "rating", "review_count", "sold_count_estimate"]].describe().T)
print("Số dòng:", len(df_clean))
print("Số duplicate theo khóa:", df_clean.duplicated(["platform", "product_id"]).sum())
print("Đã lưu:", output_path)

,product_count
platform,
lazada,3574
shopee,3771
tiki,3705


,count,mean,std,min,25%,50%,75%,max
price,10747.0,430821.195869,8.626141e+05,51.0,71000.0,165000.0,410000.0,12965000.0
original_price,3367.0,586545.196614,1.049313e+06,500.0,140000.0,270000.0,540000.0,16200000.0
discount_percent,8148.0,35.253191,1.950266e+01,1.0,20.0,35.0,48.0,100.0
rating,3110.0,4.820932,2.875274e-01,1.0,4.8,4.9,5.0,5.0
review_count,3310.0,687.515106,2.268963e+03,0.0,9.0,49.0,347.0,48201.0
sold_count_estimate,8742.0,4933.978037,2.920460e+04,1.0,12.0,125.0,997.0,970500.0


Số dòng: 11050
Số duplicate theo khóa: 0
Đã lưu: data/marketplace_products_clean.csv
